# WeaveGuard AI — Étape D : moteur PatchCore (anomalies suspectées)
**Dataset de référence : v2.** **Runtime Colab : GPU (T4 ou mieux).**

PatchCore compare les caractéristiques locales d'une image à une **mémoire de patches normaux**. Il ne donne **ni classe ni probabilité** : sa sortie est une *anomalie suspectée* avec un **score de distance** (cahier des charges §6.4, §7.2).

Ce notebook :
1. construit la mémoire à partir des recadrages « sans défaut » du **train** uniquement ;
2. **contrôle la contamination** de ces recadrages (défauts non annotés) avec une validation croisée et une revue visuelle ;
3. calibre le seuil sur la **validation** (taux de fausses alertes cible, par fenêtre de 512×512 px) ;
4. mesure détection, localisation et fausses alertes sur la validation, puis **une seule fois sur le test** ;
5. exporte la mémoire, la configuration, la calibration et une fiche modèle versionnée.

**Limites assumées** : les masques pixel n'existent pas, la localisation est comparée aux *boîtes* annotées ; aucun pli, couture ou ombre n'est annoté, donc les fausses alertes sur ces cas ne sont **pas mesurées**.

In [ ]:
# 1) Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN')
assert torch.cuda.is_available(), 'Active un GPU : Exécution > Modifier le type d exécution > T4'

In [ ]:
# 2) Configuration
from pathlib import Path
import json, hashlib, shutil, random, zipfile, datetime, time, math, collections
import numpy as np, pandas as pd, yaml
from PIL import Image, ImageOps, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from scipy import ndimage

WG = Path('/content/drive/MyDrive/WeaveGuard')
DRIVE_OUT = WG / 'datasets'
V2 = Path('/content/weaveguard_dataset_v2')
OUT = WG / 'models' / 'patchcore' / 'patchcore_v1'; OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
dev = torch.device('cuda')

BACKBONE = 'wide_resnet50_2'   # ou 'resnet50' (plus léger)
FEAT_DIM = 1024                # réduction de dimension par moyenne adaptative (layer2 + layer3 = 1536 -> 1024)
SCALE = 0.5                    # redimensionnement des images natives : un recadrage 512 px -> 256 px
CROP = 512
N_TRAIN_CROPS = 800            # recadrages utilisés pour la mémoire
CORESET_RATIO = 0.05           # part de patches conservée (greedy k-center)
PROJ_DIM = 128                 # projection aléatoire pour le coreset
BLUR_SIGMA = 4                 # lissage gaussien de la carte (px d'entrée)
FPR_TARGET = 0.05              # fausses alertes tolérées sur fenêtres normales de validation (à valider avec le métier)
MIN_REGION_AREA = 64           # aire minimale d'une région (px d'entrée, soit ~32x32 px natifs)
EXCLUDE = []                   # noms de recadrages à exclure après revue visuelle (cellule 6)
print('torch', torch.__version__, '| torchvision', torchvision.__version__)

In [ ]:
# 3) Dataset v2 en local + intégrité (hash)
if not V2.exists():
    zipfile.ZipFile(DRIVE_OUT / 'weaveguard_dataset_v2.zip').extractall(V2)
stats = json.load(open(DRIVE_OUT / 'dataset_stats_v2.json'))
man = pd.read_csv(V2 / 'manifest.csv')
DS_HASH = hashlib.sha256('\n'.join(sorted(f'{a}|{b}|{c}' for a, b, c in zip(man.name, man.md5, man.split))).encode()).hexdigest()
assert DS_HASH == stats['dataset_hash_sha256'], 'Le dataset local ne correspond pas au v2 de référence !'
NAMES = stats['classes']
for s in ('train', 'val', 'test'):
    print(s, len(list((V2 / 'images' / s).iterdir())), 'images |', len(list((V2 / 'normal_crops' / s).glob('*.jpg'))), 'recadrages sans défaut')
print('Intégrité OK, hash v2 =', DS_HASH[:16], '...')

In [ ]:
# 4) Utilitaires sans PyTorch (recadrages avec défaut, régions, appariement avec les boîtes annotées)
import numpy as np
from scipy import ndimage
from PIL import Image, ImageOps

def read_boxes(split, p, W, H):
    out = []
    for line in (V2 / 'labels' / split / (Path(p).stem + '.txt')).read_text().splitlines():
        s = line.split()
        if len(s) == 5:
            c = int(s[0]); cx, cy, bw, bh = map(float, s[1:])
            out.append((c, (cx - bw/2) * W, (cy - bh/2) * H, (cx + bw/2) * W, (cy + bh/2) * H))
    return out

def defect_windows(split, max_per_image=2):
    # fenêtres de CROP x CROP centrées sur une boîte annotée (même taille que les recadrages normaux)
    items = []
    for p in sorted((V2 / 'images' / split).iterdir()):
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); W, H = im.size
        if min(W, H) < CROP: continue
        for c, x0, y0, x1, y1 in read_boxes(split, p, W, H)[:max_per_image]:
            x = int(min(max((x0 + x1) / 2 - CROP / 2, 0), W - CROP)); y = int(min(max((y0 + y1) / 2 - CROP / 2, 0), H - CROP))
            items.append((im.crop((x, y, x + CROP, y + CROP)), NAMES[c], p.name))
    return items

def regions_from_map(m, tau, min_area):
    lab, _ = ndimage.label(m >= tau); regs = []
    for k, sl in enumerate(ndimage.find_objects(lab), 1):
        if sl is None: continue
        mask = lab[sl] == k; area = int(mask.sum())
        if area < min_area: continue
        v = m[sl][mask]
        regs.append(dict(x0=sl[1].start, y0=sl[0].start, x1=sl[1].stop, y1=sl[0].stop, area_px=area,
                         score_max=float(v.max()), score_mean=float(v.mean())))
    return regs

def rescale(regs, factor):
    return [dict(r, x0=r['x0'] * factor, y0=r['y0'] * factor, x1=r['x1'] * factor, y1=r['y1'] * factor) for r in regs]

def _inter(a, b):
    w = min(a[2], b[2]) - max(a[0], b[0]); h = min(a[3], b[3]) - max(a[1], b[1])
    return max(0.0, w) * max(0.0, h)

def match(regs, gts):
    # une boîte annotée est "retrouvée" si une région la recouvre à >= 10 % de sa surface ;
    # une région est "non annotée" si elle ne recouvre ni >= 10 % d'une boîte annotée, ni n'est couverte à >= 50 % par elle.
    hit = []
    for c, x0, y0, x1, y1 in gts:
        ga = max(1e-9, (x1 - x0) * (y1 - y0))
        hit.append((c, any(_inter((r['x0'], r['y0'], r['x1'], r['y1']), (x0, y0, x1, y1)) / ga >= 0.10 for r in regs)))
    extra = 0
    for r in regs:
        ra = max(1e-9, (r['x1'] - r['x0']) * (r['y1'] - r['y0'])); ok = False
        for c, x0, y0, x1, y1 in gts:
            i = _inter((r['x0'], r['y0'], r['x1'], r['y1']), (x0, y0, x1, y1)); ga = max(1e-9, (x1 - x0) * (y1 - y0))
            if i / ga >= 0.10 or i / ra >= 0.50: ok = True; break
        extra += 0 if ok else 1
    return hit, extra

In [ ]:
# 5) Extracteur de caractéristiques PatchCore (layer2 + layer3, agrégation locale 3x3) + mémoire
class Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        m = getattr(torchvision.models, BACKBONE)(weights='IMAGENET1K_V1')
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool); self.l1, self.l2, self.l3 = m.layer1, m.layer2, m.layer3
        for p in self.parameters(): p.requires_grad = False
    @torch.no_grad()
    def forward(self, x):
        f2 = self.l2(self.l1(self.stem(x))); f3 = self.l3(f2)
        f2 = F.avg_pool2d(f2, 3, 1, 1); f3 = F.avg_pool2d(f3, 3, 1, 1)
        f3 = F.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)
        f = torch.cat([f2, f3], 1); B, C, h, w = f.shape
        f = f.permute(0, 2, 3, 1).reshape(B * h * w, C)
        f = F.adaptive_avg_pool1d(f.unsqueeze(1), FEAT_DIM).squeeze(1)
        return f.reshape(B, h, w, FEAT_DIM)

extractor = Extractor().to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def to_tensor(im):
    # MÊME prétraitement à l'apprentissage (recadrages) et à l'inférence (images complètes) : seul SCALE intervient
    W, H = im.size; im = im.resize((round(W * SCALE), round(H * SCALE)), Image.BILINEAR)
    x = torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.).permute(2, 0, 1)
    return (x - MEAN) / STD

@torch.no_grad()
def nn_dist(Q, bank, chunk=4096):
    bn = (bank ** 2).sum(1); out = torch.empty(Q.shape[0], device=dev)
    for i in range(0, Q.shape[0], chunk):
        q = Q[i:i + chunk].float()
        d2 = (q ** 2).sum(1, keepdim=True) - 2 * q @ bank.T + bn[None]
        out[i:i + chunk] = d2.min(1).values.clamp_min(0).sqrt()
    return out

@torch.no_grad()
def greedy_coreset(X, n_select, seed=SEED):
    N = X.shape[0]; g = torch.Generator(device='cpu'); g.manual_seed(seed)
    R = (torch.randn(X.shape[1], PROJ_DIM, generator=g) / math.sqrt(PROJ_DIM)).to(dev)
    Z = torch.empty(N, PROJ_DIM, device=dev)
    for i in range(0, N, 100000): Z[i:i + 100000] = X[i:i + 100000].float() @ R
    sel = torch.empty(n_select, dtype=torch.long, device=dev)
    i0 = int(torch.randint(0, N, (1,), generator=g)); sel[0] = i0
    mind = ((Z - Z[i0]) ** 2).sum(1)
    for k in range(1, n_select):
        i = torch.argmax(mind); sel[k] = i
        mind = torch.minimum(mind, ((Z - Z[i]) ** 2).sum(1))
    return sel

all_crops = sorted((V2 / 'normal_crops' / 'train').glob('*.jpg'))
train_files = sorted(random.Random(SEED).sample(all_crops, min(N_TRAIN_CROPS, len(all_crops))))

@torch.no_grad()
def feats_for(files, bs=16):
    out = []
    for i in range(0, len(files), bs):
        x = torch.stack([to_tensor(Image.open(f).convert('RGB')) for f in files[i:i + bs]]).to(dev)
        f = extractor(x); out.append(f.reshape(f.shape[0], -1, FEAT_DIM).half())
    return torch.cat(out)

t0 = time.time(); F_train = feats_for(train_files)          # (n_crops, patches, FEAT_DIM) en fp16 sur GPU
print(f'{len(train_files)} recadrages -> {tuple(F_train.shape)} en {time.time() - t0:.0f}s | GPU : {torch.cuda.memory_allocated() / 1e9:.1f} Go')

In [ ]:
# 6) Contrôle de contamination : validation croisée sur les recadrages d'apprentissage + revue visuelle
import matplotlib.pyplot as plt
n = len(train_files); perm = np.random.RandomState(SEED).permutation(n); A, B = perm[:n // 2], perm[n // 2:]
crop_scores = np.zeros(n)
for tr, te in ((A, B), (B, A)):
    X = F_train[torch.as_tensor(tr, device=dev)].reshape(-1, FEAT_DIM)
    bank_f = X[greedy_coreset(X, int(CORESET_RATIO * X.shape[0]))].float(); del X
    for j in te: crop_scores[j] = float(nn_dist(F_train[int(j)], bank_f).max())
    torch.cuda.empty_cache()
top = np.argsort(-crop_scores)[:12]
print('Scores max par recadrage (validation croisée) : médiane %.2f | p95 %.2f | max %.2f' % (np.median(crop_scores), np.percentile(crop_scores, 95), crop_scores.max()))
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax, j in zip(axes.flat, top):
    ax.imshow(Image.open(train_files[j]).resize((256, 256))); ax.set_title(f'{train_files[j].name[:24]}\nscore {crop_scores[j]:.2f}', fontsize=7); ax.axis('off')
plt.suptitle('12 recadrages les plus atypiques : contiennent-ils un défaut visible ?'); plt.tight_layout(); plt.show()
print('Si oui, ajoute leur nom dans EXCLUDE (cellule 2), puis relance les cellules 2 à 7.')
print([train_files[j].name for j in top])

In [ ]:
# 7) Mémoire finale (coreset) + sauvegarde
keep = [i for i, f in enumerate(train_files) if f.name not in set(EXCLUDE)]
X = F_train[torch.as_tensor(keep, device=dev)].reshape(-1, FEAT_DIM)
N_PATCHES = int(X.shape[0]); n_sel = int(CORESET_RATIO * N_PATCHES)
t0 = time.time(); sel = greedy_coreset(X, n_sel); BANK = X[sel].float(); del X
torch.cuda.empty_cache()
print(f'Mémoire : {n_sel} patches retenus sur {N_PATCHES} ({CORESET_RATIO:.0%}) en {time.time() - t0:.0f}s | {len(keep)} recadrages, {len(EXCLUDE)} exclus')
MEM_CONFIG = dict(backbone=BACKBONE, weights='ImageNet IMAGENET1K_V1', layers=['layer2', 'layer3'], local_aggregation='avg_pool 3x3 stride 1',
                  feature_dim=FEAT_DIM, dim_reduction='adaptive_avg_pool1d sur les canaux', distance='L2 (plus proche voisin)',
                  memory_reduction='greedy k-center (coreset), projection aléatoire %d dim' % PROJ_DIM, coreset_ratio=CORESET_RATIO,
                  n_patches_total=N_PATCHES, n_patches_bank=n_sel, scale=SCALE, crop_px=CROP, input_norm='ImageNet mean/std',
                  blur_sigma=BLUR_SIGMA, n_train_crops=len(keep), excluded_crops=list(EXCLUDE),
                  train_crops=[train_files[i].name for i in keep], seed=SEED)
torch.save(dict(bank=BANK.half().cpu(), config=MEM_CONFIG), OUT / 'memory_bank.pt')
BANK_SHA = hashlib.sha256((OUT / 'memory_bank.pt').read_bytes()).hexdigest()
print('memory_bank.pt sha256 :', BANK_SHA[:16], '...', '| taille :', round((OUT / 'memory_bank.pt').stat().st_size / 1e6, 1), 'Mo')
del F_train; torch.cuda.empty_cache()

In [ ]:
# 8) Carte d'anomalie : score spatial (carte) et score global (max de la carte)
@torch.no_grad()
def score_map(x):
    f = extractor(x[None].to(dev)); _, h, w, d = f.shape
    dist = nn_dist(f.reshape(-1, d), BANK).reshape(1, 1, h, w)
    m = F.interpolate(dist, size=tuple(x.shape[-2:]), mode='bilinear', align_corners=False)[0, 0].cpu().numpy()
    return ndimage.gaussian_filter(m, sigma=BLUR_SIGMA)

def window_scores(items):
    return np.array([float(score_map(to_tensor(im)).max()) for im in items])

def timeit(fn, n=10):
    fn(); torch.cuda.synchronize(); t = time.time()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return (time.time() - t) / n * 1000

In [ ]:
# 9) Calibration sur la VALIDATION (fenêtres normales vs fenêtres centrées sur un défaut)
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
normal_val = [Image.open(f).convert('RGB') for f in sorted((V2 / 'normal_crops' / 'val').glob('*.jpg'))]
defect_val = defect_windows('val')
s_norm = window_scores(normal_val); s_def = window_scores([im for im, _, _ in defect_val]); cls_def = np.array([c for _, c, _ in defect_val])
auroc = roc_auc_score(np.r_[np.zeros(len(s_norm)), np.ones(len(s_def))], np.r_[s_norm, s_def])
print(f'Validation : {len(s_norm)} fenêtres normales, {len(s_def)} fenêtres avec défaut | AUROC niveau fenêtre = {auroc:.3f}')

rows = []
for fpr in (0.01, 0.02, 0.05, 0.10):
    tau_ = float(np.quantile(s_norm, 1 - fpr)); r = dict(FPR_cible=fpr, seuil=round(tau_, 3), rappel_global=round(float((s_def >= tau_).mean()), 3))
    for c in NAMES: r[c] = round(float((s_def[cls_def == c] >= tau_).mean()), 2) if (cls_def == c).any() else None
    rows.append(r)
tbl = pd.DataFrame(rows); display(tbl)
TAU = float(np.quantile(s_norm, 1 - FPR_TARGET))
print(f'>>> Seuil retenu : {TAU:.3f} (FPR cible {FPR_TARGET:.0%} sur fenêtres normales de validation). À valider avec le responsable métier (coût FP/FN, Annexe A).')

plt.figure(figsize=(8, 4)); plt.hist(s_norm, bins=30, alpha=.6, label='normales'); plt.hist(s_def, bins=30, alpha=.6, label='avec défaut')
plt.axvline(TAU, color='k', ls='--', label='seuil'); plt.xlabel('score global (distance L2, brut)'); plt.legend(); plt.title('Scores par fenêtre 512x512 (validation)'); plt.tight_layout(); plt.show()

lat_crop = timeit(lambda: score_map(to_tensor(normal_val[0])))
big = ImageOps.exif_transpose(Image.open(sorted((V2 / 'images' / 'val').iterdir())[0])).convert('RGB')
lat_full = timeit(lambda: score_map(to_tensor(big)), n=5)
print(f'Latence GPU Colab : {lat_crop:.0f} ms (fenêtre 512) | {lat_full:.0f} ms (image complète {big.size[0]}x{big.size[1]})')
CALIB = dict(tau_global=TAU, fpr_target=FPR_TARGET, auroc_val_windows=float(auroc), n_normal_val=int(len(s_norm)), n_defect_val=int(len(s_def)),
             normal_score_percentiles={str(q): float(np.percentile(s_norm, q)) for q in (50, 90, 95, 99, 100)},
             table=rows, min_region_area=MIN_REGION_AREA, latency_ms=dict(window_512=lat_crop, full_image=lat_full, gpu=torch.cuda.get_device_name(0)),
             score_meaning='distance L2 au plus proche patch normal ; échelle brute, pas une probabilité')
json.dump(CALIB, open(OUT / 'calibration.json', 'w'), indent=1)

In [ ]:
# 10) Images complètes de validation : les anomalies suspectées recouvrent-elles les défauts annotés ?
def region_eval(split, tau, return_examples=False):
    hits = collections.defaultdict(list); extras = []; flagged = 0; paths = sorted((V2 / 'images' / split).iterdir())
    for p in paths:
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); W, H = im.size
        regs = rescale(regions_from_map(score_map(to_tensor(im)), tau, MIN_REGION_AREA), 1 / SCALE)
        hit, extra = match(regs, read_boxes(split, p, W, H))
        for c, h in hit: hits[NAMES[c]].append(h)
        extras.append(extra); flagged += int(len(regs) > 0)
    res = dict(n_images=len(paths), pct_images_flagged=round(flagged / len(paths), 3), mean_unannotated_regions_per_image=round(float(np.mean(extras)), 2),
               gt_boxes_found={c: dict(n=len(v), found=round(float(np.mean(v)), 3)) for c, v in sorted(hits.items())},
               gt_boxes_found_overall=round(float(np.mean([x for v in hits.values() for x in v])), 3))
    return res
val_regions = region_eval('val', TAU)
print(json.dumps(val_regions, indent=1, ensure_ascii=False))
print('NB : "régions non annotées" = zones signalées hors des boîtes annotées : fausses alertes OU défauts non annotés (non distinguables ici).')
print('Le seuil est calibré par fenêtre 512x512 ; une image complète en contient environ 6, donc le taux d images signalées est plus élevé.')

In [ ]:
# 11) Visualisation : heatmap + régions (anomalie suspectée) + boîtes annotées
import matplotlib.pyplot as plt
vimgs = sorted((V2 / 'images' / 'val').iterdir()); sample = random.Random(SEED).sample(vimgs, 6)
fig, axes = plt.subplots(6, 2, figsize=(12, 30))
for k, p in enumerate(sample):
    im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); W, H = im.size
    m = score_map(to_tensor(im)); regs = regions_from_map(m, TAU, MIN_REGION_AREA)
    gt = im.copy(); dr = ImageDraw.Draw(gt)
    for c, x0, y0, x1, y1 in read_boxes('val', p, W, H):
        dr.rectangle([x0, y0, x1, y1], outline='lime', width=max(3, W // 250)); dr.text((x0 + 4, y0 + 4), NAMES[c], fill='lime')
    axes[k, 0].imshow(gt); axes[k, 0].set_title('vérité terrain (annotations YOLO)', fontsize=9); axes[k, 0].axis('off')
    axes[k, 1].imshow(im.resize(m.shape[::-1])); h_ = axes[k, 1].imshow(m, cmap='jet', alpha=0.45, vmin=0, vmax=max(2 * TAU, m.max()))
    for r in regs: axes[k, 1].add_patch(plt.Rectangle((r['x0'], r['y0']), r['x1'] - r['x0'], r['y1'] - r['y0'], fill=False, ec='white', lw=2))
    axes[k, 1].set_title(f'Anomalie suspectée : score de distance max {m.max():.2f} (seuil {TAU:.2f}, échelle brute). La palette n est pas une preuve de défaut.', fontsize=8); axes[k, 1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 12) Évaluation FINALE sur le TEST (une seule fois, seuil calibré sur la validation)
tfile = OUT / 'test_metrics.json'
if tfile.exists():
    print('Test déjà évalué : résultats rechargés (le test ne se ré-utilise pas).'); TEST = json.load(open(tfile))
else:
    normal_test = [Image.open(f).convert('RGB') for f in sorted((V2 / 'normal_crops' / 'test').glob('*.jpg'))]
    defect_test = defect_windows('test')
    t_norm = window_scores(normal_test); t_def = window_scores([im for im, _, _ in defect_test]); t_cls = np.array([c for _, c, _ in defect_test])
    TEST = dict(n_normal=int(len(t_norm)), n_defect=int(len(t_def)),
                auroc_windows=float(roc_auc_score(np.r_[np.zeros(len(t_norm)), np.ones(len(t_def))], np.r_[t_norm, t_def])),
                false_alarm_rate_normal_windows=float((t_norm >= TAU).mean()), recall_defect_windows=float((t_def >= TAU).mean()),
                recall_by_class={c: dict(n=int((t_cls == c).sum()), recall=round(float((t_def[t_cls == c] >= TAU).mean()), 3)) for c in NAMES if (t_cls == c).any()},
                tau=TAU, regions_full_images=region_eval('test', TAU))
    json.dump(TEST, open(tfile, 'w'), indent=1, ensure_ascii=False)
print(json.dumps(TEST, indent=1, ensure_ascii=False))

In [ ]:
# 13) Fiche modèle + rapport à coller dans le chat
card = dict(model_id='patchcore_v1', role='Détection d anomalies : anomalie suspectée + score de distance (pas de classe, pas de probabilité)',
            memory=MEM_CONFIG, memory_sha256=BANK_SHA, dataset_version='v2', dataset_hash=DS_HASH, calibration=CALIB,
            validation_regions_full_images=val_regions, test=TEST,
            labels_ui=dict(output='Anomalie suspectée : score de distance', never='ne pas afficher de pourcentage de confiance ni de classe'),
            limits=['mémoire construite sur des recadrages hors boîtes annotées de défauts : des défauts non annotés peuvent subsister',
                    'seuil calibré par fenêtre 512x512 ; le taux de fausses alertes par image complète est plus élevé',
                    'localisation comparée aux boîtes annotées, pas à des masques pixel',
                    'aucun pli, couture ou ombre annoté : fausses alertes sur ces cas non mesurées',
                    'seuil FPR à valider avec le responsable métier',
                    'latence mesurée sur GPU Colab ; à remesurer sur le matériel cible (GTX 1050, 2 Go)'],
            versions=dict(torch=torch.__version__, torchvision=torchvision.__version__, gpu=torch.cuda.get_device_name(0)),
            created=datetime.datetime.now().isoformat(timespec='seconds'))
(OUT / 'model_card.json').write_text(json.dumps(card, indent=1, ensure_ascii=False, default=str), encoding='utf-8')
print('Exporté :', OUT, '(memory_bank.pt, calibration.json, test_metrics.json, model_card.json)'); print('=' * 70)
print(json.dumps(dict(memory={k: MEM_CONFIG[k] for k in ('backbone', 'feature_dim', 'n_patches_total', 'n_patches_bank', 'n_train_crops', 'excluded_crops')},
                      calibration={k: CALIB[k] for k in ('tau_global', 'fpr_target', 'auroc_val_windows', 'normal_score_percentiles', 'latency_ms')},
                      val_table=CALIB['table'], validation_regions=val_regions, test=TEST), indent=1, ensure_ascii=False, default=str)[:9000])